# MPC Minutes 
1. Score MPC assessment of uncertainty in the economy
    1. MPC assessment of uncertainty around Inflation in India
    2. MPC assessment of uncertainty around Economic Growth in India
    3. MPC assessment of uncertainty around Unemployment in India
2. MPC Action (REPO Rate Change in Basis Points: Increase / Decrease / No Change (0))
3. Dates (Start and End) of MPC Meeting


## Inflation PMU (using Open-ai Gabriel)

In [17]:
import os
import glob
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gabriel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---------- CONFIG ----------
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

dotenv_path = Path(proj_base) / ".env"
load_dotenv(dotenv_path)

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ---------- FILE INPUT AND OUTPUT PATH ----------
pdf_folder    = os.path.join(proj_base, "data/mpc-minutes")
output_folder = os.path.join(proj_base, "output")
os.makedirs(output_folder, exist_ok=True)
output_xl     = os.path.join(output_folder, "ALL_MPC.xlsx")

# ----------  ATTRIBUTES (REFINED) ----------

attributes = {
    "start_date": (
        "From the heading at the very top of the minutes, extract the START date of the "
        "Monetary Policy Committee (MPC) meeting. The format usually appears as "
        "“Minutes of the Monetary Policy Committee Meeting, <month> <d> to <d>, <yyyy>” "
        "or similar. Use only this heading section, not later references.\n\n"
        "Return ONLY the start date as a string in dd/mm/yyyy format (e.g., 06/04/2022). "
        "Convert month names to numbers if needed. If you can infer only a single day "
        "for the meeting, treat that as both start and end for consistency. "
        "If no date can be identified, return null."
    ),

    "end_date": (
        "From the same heading at the top of the MPC minutes, extract the END date of "
        "the meeting (e.g., “April 6 to 8, 2022” → end date 08/04/2022).\n\n"
        "Return ONLY the end date as a string in dd/mm/yyyy format. If the meeting "
        "was single‑day (no range mentioned), return the same date as the start date. "
        "If no end date can be identified, return null."
    ),
    
    "repo_cut_reco": (
        "Count the NUMBER OF Monetary Policy Committee MEMBERS who voted to CUT "
        "(reduce/decrease) the policy repo rate in this meeting.\n\n"
        "Use, in order of priority:\n"
        "1) The explicit vote tally sentence in the ‘Resolution’ or ‘Voting’ section "
        "that describes how many members supported a cut, hike, or unchanged rate "
        "(e.g., “one member voted for a 25 bps reduction while five voted to keep the rate unchanged”).\n"
        "2) If needed, the individual member statements where each member clearly states "
        "their vote (e.g., “I vote for a 25 basis points reduction in the policy repo rate…”).\n\n"
        "Return an integer only (e.g., 0, 1, 2). Return 0 if no member voted for a cut "
        "or if the document indicates that all members voted only for ‘unchanged’ or ‘hike’."
    ),

    "repo_hike_reco": (
        "Count the NUMBER OF Monetary Policy Committee MEMBERS who voted to HIKE "
        "(increase/raise) the policy repo rate in this meeting.\n\n"
        "Use the same logic as for repo_cut_reco:\n"
        "1) Prefer the summary vote tally sentence in the Resolution/Voting section.\n"
        "2) If unclear, cross‑check individual member statements.\n\n"
        "Include any vote for an increase, irrespective of size (e.g., 25 bps or 50 bps). "
        "Return an integer only (e.g., 0, 3, 6). Return 0 if no member explicitly voted to hike."
    ),

    "repo_stable_reco": (
        "Count the NUMBER OF Monetary Policy Committee MEMBERS who voted to keep the "
        "policy repo rate UNCHANGED / STABLE / ON HOLD in this meeting.\n\n"
        "Again, use the official vote tally description first, then member statements:\n"
        "– Phrases like “all members voted to keep the policy repo rate unchanged at …” imply that "
        "the count equals the total number of members listed.\n"
        "– Phrases like “five members voted to keep the rate unchanged while one voted to reduce it” "
        "imply repo_stable_reco = 5.\n\n"
        "Return an integer only (e.g., 0, 5, 6). Return 0 if every member voted only for a cut or hike."
    ),

    # --- INFLATION UNCERTAINTY PERCEPTION SCORE (INF_PMU) ---
    "inf_pmu": (
        "Score the COLLECTIVE PERCEIVED UNCERTAINTY about the INFLATION OUTLOOK in this "
        "MPC minutes document on a scale from 0 to 100, following the academic practice "
        "of counting uncertainty‑related language versus confident language in central‑bank "
        "communications.\n\n"
        "Focus ONLY on uncertainty about inflation (headline, core, food, fuel) and its drivers, "
        "NOT on generic political or growth risk.\n\n"
        "Use three equal dimensions:\n"
        "(1) DATA UNCERTAINTY – missing, lagged, or conflicting inflation data; references to "
        "limited information, noisy readings, or large revisions.\n"
        "(2) PARAMETER/TRANSMISSION UNCERTAINTY – doubts about pass‑through from commodity prices, "
        "output gaps, exchange rates, or wages to inflation; references to unclear elasticities or "
        "weakened/strengthened transmission.\n"
        "(3) MODEL/FORECAST UNCERTAINTY – phrases like “highly uncertain”, “significant upside/downside "
        "risks”, “wide confidence intervals”, “unusually large forecast errors”, or emphasis on scenario "
        "bands rather than point forecasts.\n\n"
        "Heuristics:\n"
        "– 0–20: Very low inflation uncertainty – language is mostly confident, with narrow ranges and "
        "few explicit ‘uncertainty’ or ‘risk’ terms.\n"
        "– 20–50: Moderate uncertainty – mix of confident statements and repeated references to risks, "
        "uncertain pass‑through, or volatile data.\n"
        "– 50–80: High uncertainty – frequent use of strong uncertainty language and emphasis on tail risks "
        "to the inflation projection.\n"
        "– 80–100: Extreme uncertainty – inflation outlook described as highly unpredictable; multiple sources "
        "of uncertainty cited simultaneously (geopolitical shocks, commodity prices, supply chains, "
        "policy transmission, expectations de‑anchoring).\n\n"
        "Base the score on the ENTIRE document (staff assessment, outlook, and all member statements). "
        "Return a single float between 0 and 100, such as 37.5 or 72.0."
    ),

    # --- OPTIONAL: RISK BALANCE AND ANCHORING (you can drop these if not needed) ---
    "inf_risks_balance": (
        "Summarise the balance of risks around the inflation forecast as perceived in the minutes. "
        "Read the Outlook / Risks sections and member statements that talk about ‘upside risks’, "
        "‘downside risks’ or ‘risks broadly balanced’.\n\n"
        "Return ONE of the following strings only:\n"
        "– 'upside'   → if upside risks to inflation clearly dominate (more likely to overshoot).\n"
        "– 'downside' → if downside risks dominate (more likely to undershoot the projection).\n"
        "– 'balanced' → if the minutes describe risks as broadly balanced or symmetric.\n"
        "– null       → if there is not enough information to classify."
    ),

    "inf_exp_anchor": (
        "Rate how WELL‑ANCHORED inflation expectations are perceived to be in this meeting, on a 0–100 scale.\n\n"
        "Use:\n"
        "– 0–20: Expectations clearly de‑anchored (household or market expectations persistently above target, "
        "explicit concern that expectations may drift away from the target band).\n"
        "– 20–50: Partially anchored with clear upside risk; repeated concern that expectations may rise or "
        "have already moved up, but some confidence in medium‑term control.\n"
        "– 50–80: Mostly anchored; occasional references to vigilance and monitoring but baseline view is that "
        "expectations remain within the target band.\n"
        "– 80–100: Very firmly anchored; explicit statements that expectations remain well contained around the "
        "target with limited risk of de‑anchoring.\n\n"
        "Base judgment on surveys of inflation expectations, market‑based measures, and qualitative phrases in the minutes. "
        "Return a single float between 0 and 100, such as 37.5 or 72.0. or null if expectations are not discussed."
    ),
}

# ---------- ADDITIONAL INSTRUCTIONS ----------
additional_instructions = (
    "These are official minutes of the Reserve Bank of India's Monetary Policy Committee (MPC). "
    "Each document typically includes: (a) a heading with the meeting dates; (b) a Resolution section "
    "summarising the policy decision and the vote tally; (c) an Assessment/Outlook section describing "
    "inflation and growth; and (d) individual statements by each member explaining their vote.\n\n"
    "When extracting attributes, rely on the official vote summary and explicit statements about "
    "inflation risks and uncertainty wherever possible. Be consistent across documents so that "
    "date fields, vote counts, and inflation‑uncertainty scores are comparable across meetings."
)

# ---------- LOOP OVER ALL PDFs ----------
all_results = []
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files\n")

for pdf_file in pdf_files:
    fname = os.path.basename(pdf_file)
    print(f"Processing: {fname}")

    # 1) Load PDF into Gabriel dataframe
    try:
        pdf_df = gabriel.load(pdf_file, modality="pdf", reset_files=True)
    except Exception as e:
        print(f"  Load error: {e}")
        continue

    # 2) Extract attributes with Gabriel
    try:
        extract_results = await gabriel.extract(
            df                      = pdf_df,
            column_name             = "path",
            attributes              = attributes,
            additional_instructions = additional_instructions,
            save_dir                = "mpc_pdf_extract",
            model                   = "gpt-5.4",
            modality                = "pdf",
            reasoning_effort        = "high",
            reset_files             = True,
        )
    except Exception as e:
        print(f"  Extract error: {e}")
        continue

    # 3) To DataFrame
    df_file = pd.DataFrame(extract_results)
    df_file["source_file"] = fname
    print(f"  → {len(df_file)} articles extracted")
    all_results.append(df_file)

# ---------- COMBINE AND SAVE ----------
if not all_results:
    print("No data extracted.")
else:
    combined_df = pd.concat(all_results, ignore_index=True)

    # Drop internal Gabriel columns if present
    gabriel_cols = ["path", "entity_name", "entity_id"]
    combined_df = combined_df.drop(
        columns=[c for c in gabriel_cols if c in combined_df.columns],
        errors="ignore",
    )

    # Keep only intended columns (those that actually exist)
    intended_cols = list(attributes.keys()) + ["source_file"]
    combined_df = combined_df[[c for c in intended_cols if c in combined_df.columns]]

    # Convert numeric types
    for col in ["inf_pmu", "inf_expectations_anchor"]:
        if col in combined_df.columns:
            combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce")

    # Save to Excel
    combined_df.to_excel(output_xl, index=False)
    print(f"\nSaved {len(combined_df)} meetings → {output_xl}")
    print("Columns:", list(combined_df.columns))


Found 59 PDF files

Processing: MPC April 2017.pdf
[gabriel.load] PDF modality attaches PDFs directly (richer layout, figures, and images). Set modality='text' (or 'entity'/'web') to extract text-only versions of PDFs.
                 name                                               path
0  MPC April 2017.pdf  /Users/kalyan/Library/CloudStorage/OneDrive-Pe...
Saved aggregated file to /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/data/mpc-minutes/gabriel_aggregated_content.csv
[Extract] Rendering 1 prompts…
Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~1,537 | Words per prompt: ~1,537
Model: gpt-5.4 | Reasoning effort: high | Mode: streaming | modality: pdf
Pricing for model 'gpt-5.4': input $2.5/1M, output $15.0/1M
Estimated token usage: input 4,305, output 500 | ~5,305 tokens per call
Estimated synchronous cost: $0.02 (input: $0.01, output: $0.01)
Note: multimedia/web inputs can make c

Processing prompts:   0%|                                 | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-03-25 00:50:57 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=129 prompts/min, cap=27, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 26 to 128 parallel threads over 15s.
[parallelization] Ramp-up complete at 128 parallel threads.
[dynamic timeout] Initialized timeout to 199.2s (p90=79.7s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 12,864 tokens, output (incl. reasoning) ≈ 3,625 tokens.
[token estimate] Updated estimated total cost: ~$0.09 (input $0.03, output $0.05). Updated parallel threads: 26 based on refreshed token usage.
[token estimate] Note: multimedia/web inputs can make cost estimates unreliable. Monitor usage in the OpenAI dashboard.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher us

Processing prompts:   0%|                                 | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-03-25 00:52:23 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=129 prompts/min, cap=27, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 26 to 128 parallel threads over 15s.
[parallelization] Ramp-up complete at 128 parallel threads.
[dynamic timeout] Initialized timeout to 91.9s (p90=36.7s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 14,385 tokens, output (incl. reasoning) ≈ 4,256 tokens.
[token estimate] Updated estimated total cost: ~$0.10 (input $0.04, output $0.06). Updated parallel threads: 23 based on refreshed token usage.
[token estimate] Note: multimedia/web inputs can make cost estimates unreliable. Monitor usage in the OpenAI dashboard.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usa

...

[parallelization] 2026-03-25 01:36:59 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=129 prompts/min, cap=27, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 26 to 128 parallel threads over 15s.
[parallelization] Ramp-up complete at 128 parallel threads.
[dynamic timeout] Initialized timeout to 54.7s (p90=21.9s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 14,940 tokens, output (incl. reasoning) ≈ 2,183 tokens.
[token estimate] Updated estimated total cost: ~$0.07 (input $0.04, output $0.03). Updated parallel threads: 25 based on refreshed token usage.
[token estimate] Note: multimedia/web inputs can make cost estimates unreliable. Monitor usage in the OpenAI dashboard.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usa

Processing prompts:   0%|                                 | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-03-25 01:37:25 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=129 prompts/min, cap=27, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 26 to 128 parallel threads over 15s.
[parallelization] Ramp-up complete at 128 parallel threads.
[dynamic timeout] Initialized timeout to 96.2s (p90=38.5s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 22,395 tokens, output (incl. reasoning) ≈ 4,815 tokens.
[token estimate] Updated estimated total cost: ~$0.13 (input $0.06, output $0.07). Updated parallel threads: 16 based on refreshed token usage.
[token estimate] Note: multimedia/web inputs can make cost estimates unreliable. Monitor usage in the OpenAI dashboard.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usa

Processing prompts:   0%|                                 | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-03-25 01:38:08 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=129 prompts/min, cap=27, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 26 to 128 parallel threads over 15s.
[parallelization] Ramp-up complete at 128 parallel threads.
[dynamic timeout] Initialized timeout to 37.7s (p90=15.1s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 17,342 tokens, output (incl. reasoning) ≈ 1,155 tokens.
[token estimate] Updated estimated total cost: ~$0.06 (input $0.04, output $0.02). Updated parallel threads: 23 based on refreshed token usage.
[token estimate] Note: multimedia/web inputs can make cost estimates unreliable. Monitor usage in the OpenAI dashboard.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usa

In [25]:
combined_df.sort_values('start_date')
combined_df


,start_date,end_date,repo_cut_reco,repo_hike_reco,repo_stable_reco,inf_pmu,inf_risks_balance,inf_exp_anchor,source_file
0,05/04/2017,06/04/2017,0,0,6,60.0,balanced,35.0,MPC April 2017.pdf
1,04/04/2018,05/04/2018,0,1,5,70.0,upside,35.0,MPC April 2018.pdf
2,02/04/2019,04/04/2019,4,0,2,67.5,balanced,60.0,MPC April 2019.pdf
3,24/03/2020,27/03/2020,6,0,0,74.0,downside,62.5,MPC April 2020.pdf
4,05/04/2021,07/04/2021,0,0,6,42.5,balanced,72.0,MPC April 2021.pdf
5,06/04/2022,08/04/2022,0,0,6,76.0,upside,42.0,MPC April 2022.pdf
6,03/04/2023,06/04/2023,0,0,6,70.0,balanced,65.0,MPC April 2023.pdf
7,03/04/2024,05/04/2024,1,0,5,62.5,balanced,72.0,MPC April 2024.pdf
8,07/04/2025,09/04/2025,6,0,0,32.0,balanced,82.0,MPC April 2025.pdf
9,01/08/2017,02/08/2017,5,0,1,67.5,upside,37.5,MPC Aug 2017.pdf


### Word2Vec on MPC minutes

In [11]:
# ============================================
# RBI MPC Minutes: Word2Vec-based uncertainty
# and policy stance analysis
# ============================================

# ---------- CONFIG ----------
import os
import re
import json
import math
import glob
import warnings
from collections import Counter, defaultdict

proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

# ---------- FILE INPUT AND OUTPUT PATH ----------
pdf_folder    = os.path.join(proj_base, "data/mpc-minutes")
output_folder = os.path.join(proj_base, "output/mpc")
os.makedirs(output_folder, exist_ok=True)

warnings.filterwarnings("ignore")

# ---------- INSTALL NOTES ----------
# pip install pymupdf pandas numpy matplotlib seaborn gensim scikit-learn nltk python-dateutil

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dateutil import parser as dtparser
from gensim.models import Word2Vec
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

import nltk
nltk.download("punkt")
nltk.download("stopwords")
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

# --------------------------------------------
# 1. PDF TEXT EXTRACTION
# --------------------------------------------
def extract_text_from_pdf(pdf_path):
    text_parts = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text_parts.append(page.get_text("text"))
    doc.close()
    return "\n".join(text_parts)

# --------------------------------------------
# 2. CLEANING
# --------------------------------------------
def normalize_text(text):
    text = text.replace("\x0c", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)
    return text.strip()

def basic_clean_for_metadata(text):
    return normalize_text(text)

def clean_for_tokenization(text):
    text = text.lower()
    text = re.sub(r"\d+(\.\d+)?", " NUM ", text)
    text = re.sub(r"[^a-zA-Z\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# --------------------------------------------
# 3. DATE EXTRACTION
# --------------------------------------------
MONTH_RE = r"(January|February|March|April|May|June|July|August|September|October|November|December)"

def extract_meeting_date(text, filename=None):
    candidates = []

    patterns = [
        rf"meeting\s+(?:held|during)\s+(?:on\s+)?((?:{MONTH_RE}\s+\d{{1,2}},\s*\d{{4}})|(?:{MONTH_RE}\s+\d{{1,2}}\s*(?:,|-|and)\s*\d{{1,2}}(?:,)?\s*\d{{4}}))",
        rf"Minutes of the Monetary Policy Committee Meeting\s+((?:{MONTH_RE}\s+\d{{1,2}},?\s*(?:and\s+)?\d{{1,2}},?\s*\d{{4}})|(?:{MONTH_RE}\s+\d{{1,2}}\s*,\s*\d{{4}}))",
        rf"held during\s+((?:{MONTH_RE}.*?\d{{4}}))"
    ]

    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            candidates.append(m.group(1))

    if filename:
        m2 = re.search(r"(20\d{2})[-_]?([01]?\d)[-_]?([0-3]?\d)?", filename) #Extract Date using dd/mm/yyyy pattern
        if m2:
            y, mo, d = m2.groups()
            if d:
                try:
                    candidates.append(f"{y}-{mo}-{d}")
                except:
                    pass

    for c in candidates:
        try:
            d = dtparser.parse(c, fuzzy=True, dayfirst=False)
            return pd.Timestamp(d.date())
        except:
            continue

    return pd.NaT

# --------------------------------------------
# 4. SPEAKER SEGMENTATION
# --------------------------------------------
def split_speaker_sections(text):
    """
    Tries to isolate 'Statement by ...' sections, common in RBI MPC minutes.
    Falls back to whole document if no sections found.
    """
    pattern = r"(Statement by\s+[A-Z][A-Za-z\.\s\-]+)"
    matches = list(re.finditer(pattern, text))

    sections = []
    if not matches:
        sections.append({"speaker": "FULL_DOCUMENT", "text": text})
        return sections

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        header = m.group(1).strip()
        speaker = re.sub(r"^Statement by\s+", "", header).strip()
        chunk = text[start:end].strip()
        sections.append({"speaker": speaker, "text": chunk})

    return sections

# --------------------------------------------
# 5. TOKENIZATION FOR WORD2VEC
# --------------------------------------------
def tokenize_for_w2v(text):
    text = clean_for_tokenization(text)
    sents = sent_tokenize(text)
    tokenized = []
    for s in sents:
        tokens = [w for w in word_tokenize(s) if w.isalpha()]
        tokens = [w for w in tokens if w not in STOPWORDS and len(w) > 2]
        if len(tokens) >= 3:
            tokenized.append(tokens)
    return tokenized

# --------------------------------------------
# 6. LOAD ALL PDFS
# --------------------------------------------
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files.")

docs = []
all_sentences = []

for pdf_path in pdf_files:
    raw_text = extract_text_from_pdf(pdf_path)
    meta_text = basic_clean_for_metadata(raw_text)
    filename = os.path.basename(pdf_path)
    meeting_date = extract_meeting_date(meta_text, filename=filename)

    speaker_sections = split_speaker_sections(meta_text)

    for sec in speaker_sections:
        tokenized_sentences = tokenize_for_w2v(sec["text"])
        all_sentences.extend(tokenized_sentences)

        docs.append({
            "file_name": filename,
            "file_path": pdf_path,
            "meeting_date": meeting_date,
            "speaker": sec["speaker"],
            "raw_text": sec["text"],
            "clean_text": clean_for_tokenization(sec["text"]),
            "num_sentences": len(tokenized_sentences),
            "num_tokens": sum(len(x) for x in tokenized_sentences)
        })

docs_df = pd.DataFrame(docs)
docs_df.to_csv(os.path.join(output_folder, "mpc_extracted_sections.csv"), index=False)
print("Saved extracted sections.")

# --------------------------------------------
# 7. TRAIN WORD2VEC
# --------------------------------------------
# Start with Skip-gram (sg=1), as it is often better for semantics
# You can switch to CBOW with sg=0
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # 1 = Skip-gram, 0 = CBOW
    epochs=100,
    negative=10,
    sample=1e-3
)

w2v_model.save(os.path.join(output_folder, "mpc_word2vec.model"))
print("Word2Vec model trained and saved.")

# --------------------------------------------
# 8. SEED LISTS
# --------------------------------------------
uncertainty_seeds = [
    "uncertainty", "risk", "risks", "volatile", "volatility", "shock", "shocks",
    "fragile", "unknown", "unclear", "adverse", "downside", "scenario", "scenarios"
]

data_model_seeds = [
    "data", "forecast", "forecasts", "projection", "projections", "estimate", "estimates",
    "revision", "revisions", "model", "models", "parameter", "parameters", "assumption",
    "assumptions", "output", "gap", "pass", "through"
]

hawkish_seeds = [
    "inflation", "persistent", "pressure", "pressures", "vigilance", "tightening",
    "anchor", "anchoring", "contain", "stability"
]

dovish_seeds = [
    "growth", "slowdown", "support", "accommodative", "easing", "stimulus",
    "recovery", "downside", "liquidity"
]

policy_stance_words = ["accommodative", "neutral", "withdrawal", "tightening", "easing"]

# --------------------------------------------
# 9. EXPAND SEEDS USING EMBEDDINGS
# --------------------------------------------
def vocab_filter(words, model):
    return [w for w in words if w in model.wv.key_to_index]

def expand_seed_list(seed_words, model, topn=10):
    seed_words = vocab_filter(seed_words, model)
    expanded = set(seed_words)
    for w in seed_words:
        try:
            sims = model.wv.most_similar(w, topn=topn)
            for term, score in sims:
                if score >= 0.45:
                    expanded.add(term)
        except:
            pass
    return sorted(expanded)

uncertainty_terms = expand_seed_list(uncertainty_seeds, w2v_model, topn=15)
data_model_terms = expand_seed_list(data_model_seeds, w2v_model, topn=15)
hawkish_terms = expand_seed_list(hawkish_seeds, w2v_model, topn=15)
dovish_terms = expand_seed_list(dovish_seeds, w2v_model, topn=15)

with open(os.path.join(output_folder, "expanded_term_sets.json"), "w") as f:
    json.dump({
        "uncertainty_terms": uncertainty_terms,
        "data_model_terms": data_model_terms,
        "hawkish_terms": hawkish_terms,
        "dovish_terms": dovish_terms
    }, f, indent=2)

# --------------------------------------------
# 10. DOCUMENT SCORING
# --------------------------------------------
def count_terms(text, terms):
    tokens = text.split()
    ctr = Counter(tokens)
    return sum(ctr[t] for t in terms if t in ctr)

def relative_term_score(text, terms):
    tokens = text.split()
    n = max(len(tokens), 1)
    return count_terms(text, terms) / n * 1000  # per 1000 tokens

def avg_similarity_to_seedset(doc_tokens, seed_words, model):
    seed_words = vocab_filter(seed_words, model)
    doc_tokens = [t for t in doc_tokens if t in model.wv.key_to_index]
    if not seed_words or not doc_tokens:
        return np.nan

    sims = []
    for t in doc_tokens:
        vals = []
        for s in seed_words:
            try:
                vals.append(model.wv.similarity(t, s))
            except:
                continue
        if vals:
            sims.append(np.mean(vals))
    return np.mean(sims) if sims else np.nan

rows = []
for _, r in docs_df.iterrows():
    doc_tokens = [t for t in r["clean_text"].split() if len(t) > 2]

    uncertainty_freq = relative_term_score(r["clean_text"], uncertainty_terms)
    data_model_freq = relative_term_score(r["clean_text"], data_model_terms)
    hawkish_freq = relative_term_score(r["clean_text"], hawkish_terms)
    dovish_freq = relative_term_score(r["clean_text"], dovish_terms)

    uncertainty_embed = avg_similarity_to_seedset(doc_tokens, uncertainty_seeds, w2v_model)
    data_model_embed = avg_similarity_to_seedset(doc_tokens, data_model_seeds, w2v_model)
    hawkish_embed = avg_similarity_to_seedset(doc_tokens, hawkish_seeds, w2v_model)
    dovish_embed = avg_similarity_to_seedset(doc_tokens, dovish_seeds, w2v_model)

    policy_bias_freq = hawkish_freq - dovish_freq
    policy_bias_embed = (hawkish_embed if pd.notna(hawkish_embed) else 0) - (dovish_embed if pd.notna(dovish_embed) else 0)

    rows.append({
        "file_name": r["file_name"],
        "meeting_date": r["meeting_date"],
        "speaker": r["speaker"],
        "num_tokens": r["num_tokens"],
        "uncertainty_freq": uncertainty_freq,
        "data_model_freq": data_model_freq,
        "hawkish_freq": hawkish_freq,
        "dovish_freq": dovish_freq,
        "uncertainty_embed": uncertainty_embed,
        "data_model_embed": data_model_embed,
        "hawkish_embed": hawkish_embed,
        "dovish_embed": dovish_embed,
        "policy_bias_freq": policy_bias_freq,
        "policy_bias_embed": policy_bias_embed
    })

scores_df = pd.DataFrame(rows)

# --------------------------------------------
# 11. OPTIONAL: MAP POLICY STANCE LABELS
# --------------------------------------------
# You should ideally create this from actual policy decisions.
# For now, infer rough stance from text if available.

def infer_stance(text):
    t = text.lower()
    if "accommodative stance" in t:
        return "accommodative"
    elif "neutral stance" in t:
        return "neutral"
    elif "tightening" in t or "withdrawal of accommodation" in t:
        return "hawkish"
    else:
        return "unknown"

full_doc_text = docs_df.groupby(["file_name", "meeting_date"], as_index=False)["raw_text"].apply(lambda x: " ".join(x)).reset_index(drop=True)
full_doc_text["stance_label"] = full_doc_text["raw_text"].apply(infer_stance)

meeting_scores = scores_df.groupby(["file_name", "meeting_date"], as_index=False).agg({
    "uncertainty_freq":"mean",
    "data_model_freq":"mean",
    "hawkish_freq":"mean",
    "dovish_freq":"mean",
    "uncertainty_embed":"mean",
    "data_model_embed":"mean",
    "hawkish_embed":"mean",
    "dovish_embed":"mean",
    "policy_bias_freq":"mean",
    "policy_bias_embed":"mean",
    "num_tokens":"sum"
})

meeting_scores = meeting_scores.merge(
    full_doc_text[["file_name", "meeting_date", "stance_label"]],
    on=["file_name", "meeting_date"],
    how="left"
)

# --------------------------------------------
# 12. STANDARDIZED INDICES
# --------------------------------------------
for col in ["uncertainty_freq", "data_model_freq", "uncertainty_embed", "data_model_embed", "policy_bias_freq", "policy_bias_embed"]:
    zcol = col + "_z"
    vals = meeting_scores[col].astype(float)
    meeting_scores[zcol] = (vals - vals.mean()) / (vals.std(ddof=0) if vals.std(ddof=0) != 0 else 1)

meeting_scores["composite_uncertainty_index"] = (
    0.5 * meeting_scores["uncertainty_freq_z"] +
    0.5 * meeting_scores["uncertainty_embed_z"]
)

meeting_scores["composite_data_model_uncertainty"] = (
    0.5 * meeting_scores["data_model_freq_z"] +
    0.5 * meeting_scores["data_model_embed_z"]
)

meeting_scores["composite_policy_bias"] = (
    0.5 * meeting_scores["policy_bias_freq_z"] +
    0.5 * meeting_scores["policy_bias_embed_z"]
)

# --------------------------------------------
# 13. SIMPLE IMPACT ANALYSIS
# --------------------------------------------
# Numeric proxy for stance:
stance_map = {
    "hawkish": 1,
    "neutral": 0,
    "accommodative": -1,
    "unknown": np.nan
}
meeting_scores["stance_num"] = meeting_scores["stance_label"].map(stance_map)

reg_df = meeting_scores.dropna(subset=["stance_num", "composite_uncertainty_index", "composite_data_model_uncertainty"]).copy()

if len(reg_df) >= 5:
    X = reg_df[["composite_uncertainty_index", "composite_data_model_uncertainty"]]
    y = reg_df["stance_num"]

    reg = LinearRegression()
    reg.fit(X, y)

    coef_df = pd.DataFrame({
        "variable": X.columns,
        "coefficient": reg.coef_
    })
    coef_df["intercept"] = reg.intercept_
else:
    coef_df = pd.DataFrame(columns=["variable", "coefficient", "intercept"])

# --------------------------------------------
# 14. EXPORT TABLES
# --------------------------------------------
scores_df.to_csv(os.path.join(output_folder, "speaker_level_scores.csv"), index=False)
meeting_scores.to_csv(os.path.join(output_folder, "meeting_level_scores.csv"), index=False)
coef_df.to_csv(os.path.join(output_folder, "stance_regression_coefficients.csv"), index=False)

# --------------------------------------------
# 15. PLOTS
# --------------------------------------------
sns.set(style="whitegrid")

# Sort by date
meeting_scores = meeting_scores.sort_values("meeting_date")

# Plot 1: Uncertainty over time
plt.figure(figsize=(12, 5))
plt.plot(meeting_scores["meeting_date"], meeting_scores["composite_uncertainty_index"], marker="o", label="Composite uncertainty")
plt.plot(meeting_scores["meeting_date"], meeting_scores["composite_data_model_uncertainty"], marker="o", label="Data/model uncertainty")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.title("RBI MPC Minutes: Uncertainty Indices Over Time")
plt.xlabel("Meeting date")
plt.ylabel("Standardized index")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "uncertainty_indices_over_time.png"), dpi=300)
plt.close()

# Plot 2: Policy bias vs uncertainty
plt.figure(figsize=(8, 6))
sns.regplot(
    data=meeting_scores,
    x="composite_uncertainty_index",
    y="composite_policy_bias",
    scatter_kws={"s": 70}
)
plt.title("Policy Bias vs Perceived Uncertainty")
plt.xlabel("Composite uncertainty index")
plt.ylabel("Composite policy bias")
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "policy_bias_vs_uncertainty.png"), dpi=300)
plt.close()

# Plot 3: Speaker heterogeneity
speaker_summary = scores_df.groupby("speaker", as_index=False).agg({
    "uncertainty_freq":"mean",
    "data_model_freq":"mean",
    "policy_bias_freq":"mean",
    "num_tokens":"sum"
})
speaker_summary = speaker_summary[speaker_summary["speaker"] != "FULL_DOCUMENT"].sort_values("uncertainty_freq", ascending=False)

if len(speaker_summary) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=speaker_summary, x="uncertainty_freq", y="speaker", palette="viridis")
    plt.title("Average Uncertainty Language by Speaker")
    plt.xlabel("Uncertainty-term frequency per 1000 tokens")
    plt.ylabel("Speaker")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "speaker_uncertainty_bar.png"), dpi=300)
    plt.close()

print("Done. Outputs saved to:", output_folder)

[nltk_data] Downloading package punkt to /Users/kalyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kalyan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Found 59 PDF files.
Saved extracted sections.
Word2Vec model trained and saved.
Done. Outputs saved to: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/output/mpc


In [3]:
# ============================================
# RBI MPC Minutes: Word2Vec-based uncertainty
# and policy stance analysis
# ============================================

# PLAIN ENGLISH EXPLANATION
# ------------------------------------------------------------
# This code reads all RBI MPC minutes PDF files from a folder.
# For each PDF, it extracts the full text and looks for the
# header line that usually says:
# "Minutes of the Monetary Policy Committee Meeting, Month DD to DD, YYYY"
#
# From that header, it extracts:
# 1. start_date in dd/mm/yyyy format
# 2. end_date in dd/mm/yyyy format
#
# Then it:
# - splits the minutes into speaker sections when possible
# - cleans and tokenizes the text
# - trains a Word2Vec model on the corpus
# - builds word lists related to uncertainty, data/model uncertainty,
#   hawkishness, and dovishness
# - scores each speaker section and each meeting
# - creates summary outputs
# - saves outputs as Excel (.xlsx) files instead of CSV
#
# PSEUDOCODE
# ------------------------------------------------------------
# 1. Define input and output folders
# 2. Read each PDF file
# 3. Extract text from all pages
# 4. Find the MPC header and extract start_date and end_date
# 5. Split document into speaker sections
# 6. Clean and tokenize text for Word2Vec
# 7. Train Word2Vec model
# 8. Expand seed word lists using the trained model
# 9. Score each speaker section for uncertainty and policy stance
# 10. Aggregate section scores to the meeting level
# 11. Infer a rough policy stance label from the text
# 12. Create standardized indices
# 13. Run a simple regression
# 14. Save outputs as .xlsx files
# 15. Save charts as .png files

# ---------- CONFIG ----------
import os
import re
import json
import math
import glob
import warnings
import calendar
from collections import Counter, defaultdict

proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

# ---------- FILE INPUT AND OUTPUT PATH ----------
pdf_folder    = os.path.join(proj_base, "data/mpc-minutes")
output_folder = os.path.join(proj_base, "output/mpc")
os.makedirs(output_folder, exist_ok=True)

warnings.filterwarnings("ignore")

# ---------- INSTALL NOTES ----------
# pip install pymupdf pandas numpy matplotlib seaborn gensim scikit-learn nltk python-dateutil openpyxl

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dateutil import parser as dtparser
from gensim.models import Word2Vec
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

import nltk
nltk.download("punkt")
nltk.download("stopwords")
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

# --------------------------------------------
# 1. PDF TEXT EXTRACTION
# --------------------------------------------
def extract_text_from_pdf(pdf_path):
    text_parts = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text_parts.append(page.get_text("text"))
    doc.close()
    return "\n".join(text_parts)

# --------------------------------------------
# 2. CLEANING
# --------------------------------------------
def normalize_text(text):
    text = text.replace("\x0c", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)
    return text.strip()

def basic_clean_for_metadata(text):
    return normalize_text(text)

def clean_for_tokenization(text):
    text = text.lower()
    text = re.sub(r"\d+(\.\d+)?", " NUM ", text)
    text = re.sub(r"[^a-zA-Z\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# --------------------------------------------
# 3. DATE EXTRACTION FROM MPC HEADER
# --------------------------------------------
def extract_mpc_date_range(text):
    """
    Expected header pattern:
    'Minutes of the Monetary Policy Committee Meeting, April 6 to 8, 2022'

    Returns:
        start_date_str -> dd/mm/yyyy
        end_date_str   -> dd/mm/yyyy
        start_date_ts  -> pandas Timestamp
        end_date_ts    -> pandas Timestamp
    """
    text = normalize_text(text)

    pattern = (
        r"Minutes of the Monetary Policy Committee Meeting,\s*"
        r"([A-Za-z]+)\s+(\d{1,2})\s+(?:to|-)\s+(\d{1,2}),\s*(\d{4})"
    )

    m = re.search(pattern, text, flags=re.IGNORECASE)
    if not m:
        return None, None, pd.NaT, pd.NaT

    month_name, start_day, end_day, year = m.groups()
    month_name = month_name.strip()
    start_day = int(start_day)
    end_day = int(end_day)
    year = int(year)

    try:
        month_num = list(calendar.month_name).index(month_name.capitalize())
        start_date_ts = pd.Timestamp(year=year, month=month_num, day=start_day)
        end_date_ts = pd.Timestamp(year=year, month=month_num, day=end_day)

        start_date_str = start_date_ts.strftime("%d/%m/%Y")
        end_date_str = end_date_ts.strftime("%d/%m/%Y")

        return start_date_str, end_date_str, start_date_ts, end_date_ts
    except:
        return None, None, pd.NaT, pd.NaT

# --------------------------------------------
# 4. SPEAKER SEGMENTATION
# --------------------------------------------
def split_speaker_sections(text):
    """
    Tries to isolate 'Statement by ...' sections, common in RBI MPC minutes.
    Falls back to whole document if no sections found.
    """
    pattern = r"(Statement by\s+[A-Z][A-Za-z\.\s\-]+)"
    matches = list(re.finditer(pattern, text))

    sections = []
    if not matches:
        sections.append({"speaker": "FULL_DOCUMENT", "text": text})
        return sections

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        header = m.group(1).strip()
        speaker = re.sub(r"^Statement by\s+", "", header).strip()
        chunk = text[start:end].strip()
        sections.append({"speaker": speaker, "text": chunk})

    return sections

# --------------------------------------------
# 5. TOKENIZATION FOR WORD2VEC
# --------------------------------------------
def tokenize_for_w2v(text):
    text = clean_for_tokenization(text)
    sents = sent_tokenize(text)
    tokenized = []
    for s in sents:
        tokens = [w for w in word_tokenize(s) if w.isalpha()]
        tokens = [w for w in tokens if w not in STOPWORDS and len(w) > 2]
        if len(tokens) >= 3:
            tokenized.append(tokens)
    return tokenized

# --------------------------------------------
# 6. LOAD ALL PDFS
# --------------------------------------------
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files.")

docs = []
all_sentences = []

for pdf_path in pdf_files:
    raw_text = extract_text_from_pdf(pdf_path)
    meta_text = basic_clean_for_metadata(raw_text)
    filename = os.path.basename(pdf_path)

    start_date, end_date, start_date_ts, end_date_ts = extract_mpc_date_range(meta_text)

    speaker_sections = split_speaker_sections(meta_text)

    for sec in speaker_sections:
        tokenized_sentences = tokenize_for_w2v(sec["text"])
        all_sentences.extend(tokenized_sentences)

        docs.append({
            "file_name": filename,
            "file_path": pdf_path,
            "start_date": start_date,
            "end_date": end_date,
            "start_date_ts": start_date_ts,
            "end_date_ts": end_date_ts,
            "speaker": sec["speaker"],
            "raw_text": sec["text"],
            "clean_text": clean_for_tokenization(sec["text"]),
            "num_sentences": len(tokenized_sentences),
            "num_tokens": sum(len(x) for x in tokenized_sentences)
        })

docs_df = pd.DataFrame(docs)
docs_df.to_excel(os.path.join(output_folder, "mpc_extracted_sections.xlsx"), index=False)
print("Saved extracted sections.")

# --------------------------------------------
# 7. TRAIN WORD2VEC
# --------------------------------------------
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # 1 = Skip-gram, 0 = CBOW
    epochs=100,
    negative=10,
    sample=1e-3
)

w2v_model.save(os.path.join(output_folder, "mpc_word2vec.model"))
print("Word2Vec model trained and saved.")

# --------------------------------------------
# 8. SEED LISTS
# --------------------------------------------
uncertainty_seeds = [
    "uncertainty", "risk", "risks", "volatile", "volatility", "shock", "shocks",
    "fragile", "unknown", "unclear", "adverse", "downside", "scenario", "scenarios"
]

data_model_seeds = [
    "data", "forecast", "forecasts", "projection", "projections", "estimate", "estimates",
    "revision", "revisions", "model", "models", "parameter", "parameters", "assumption",
    "assumptions", "output", "gap", "pass", "through"
]

hawkish_seeds = [
    "inflation", "persistent", "pressure", "pressures", "vigilance", "tightening",
    "anchor", "anchoring", "contain", "stability"
]

dovish_seeds = [
    "growth", "slowdown", "support", "accommodative", "easing", "stimulus",
    "recovery", "downside", "liquidity"
]

policy_stance_words = ["accommodative", "neutral", "withdrawal", "tightening", "easing"]

# --------------------------------------------
# 9. EXPAND SEEDS USING EMBEDDINGS
# --------------------------------------------
def vocab_filter(words, model):
    return [w for w in words if w in model.wv.key_to_index]

def expand_seed_list(seed_words, model, topn=10):
    seed_words = vocab_filter(seed_words, model)
    expanded = set(seed_words)
    for w in seed_words:
        try:
            sims = model.wv.most_similar(w, topn=topn)
            for term, score in sims:
                if score >= 0.45:
                    expanded.add(term)
        except:
            pass
    return sorted(expanded)

uncertainty_terms = expand_seed_list(uncertainty_seeds, w2v_model, topn=15)
data_model_terms = expand_seed_list(data_model_seeds, w2v_model, topn=15)
hawkish_terms = expand_seed_list(hawkish_seeds, w2v_model, topn=15)
dovish_terms = expand_seed_list(dovish_seeds, w2v_model, topn=15)

with open(os.path.join(output_folder, "expanded_term_sets.json"), "w") as f:
    json.dump({
        "uncertainty_terms": uncertainty_terms,
        "data_model_terms": data_model_terms,
        "hawkish_terms": hawkish_terms,
        "dovish_terms": dovish_terms
    }, f, indent=2)

# --------------------------------------------
# 10. DOCUMENT SCORING
# --------------------------------------------
def count_terms(text, terms):
    tokens = text.split()
    ctr = Counter(tokens)
    return sum(ctr[t] for t in terms if t in ctr)

def relative_term_score(text, terms):
    tokens = text.split()
    n = max(len(tokens), 1)
    return count_terms(text, terms) / n * 1000

def avg_similarity_to_seedset(doc_tokens, seed_words, model):
    seed_words = vocab_filter(seed_words, model)
    doc_tokens = [t for t in doc_tokens if t in model.wv.key_to_index]
    if not seed_words or not doc_tokens:
        return np.nan

    sims = []
    for t in doc_tokens:
        vals = []
        for s in seed_words:
            try:
                vals.append(model.wv.similarity(t, s))
            except:
                continue
        if vals:
            sims.append(np.mean(vals))
    return np.mean(sims) if sims else np.nan

rows = []
for _, r in docs_df.iterrows():
    doc_tokens = [t for t in r["clean_text"].split() if len(t) > 2]

    uncertainty_freq = relative_term_score(r["clean_text"], uncertainty_terms)
    data_model_freq = relative_term_score(r["clean_text"], data_model_terms)
    hawkish_freq = relative_term_score(r["clean_text"], hawkish_terms)
    dovish_freq = relative_term_score(r["clean_text"], dovish_terms)

    uncertainty_embed = avg_similarity_to_seedset(doc_tokens, uncertainty_seeds, w2v_model)
    data_model_embed = avg_similarity_to_seedset(doc_tokens, data_model_seeds, w2v_model)
    hawkish_embed = avg_similarity_to_seedset(doc_tokens, hawkish_seeds, w2v_model)
    dovish_embed = avg_similarity_to_seedset(doc_tokens, dovish_seeds, w2v_model)

    policy_bias_freq = hawkish_freq - dovish_freq
    policy_bias_embed = (hawkish_embed if pd.notna(hawkish_embed) else 0) - (dovish_embed if pd.notna(dovish_embed) else 0)

    rows.append({
        "file_name": r["file_name"],
        "start_date": r["start_date"],
        "end_date": r["end_date"],
        "speaker": r["speaker"],
        "num_tokens": r["num_tokens"],
        "uncertainty_freq": uncertainty_freq,
        "data_model_freq": data_model_freq,
        "hawkish_freq": hawkish_freq,
        "dovish_freq": dovish_freq,
        "uncertainty_embed": uncertainty_embed,
        "data_model_embed": data_model_embed,
        "hawkish_embed": hawkish_embed,
        "dovish_embed": dovish_embed,
        "policy_bias_freq": policy_bias_freq,
        "policy_bias_embed": policy_bias_embed
    })

scores_df = pd.DataFrame(rows)

# --------------------------------------------
# 11. OPTIONAL: MAP POLICY STANCE LABELS
# --------------------------------------------
def infer_stance(text):
    t = text.lower()
    if "accommodative stance" in t:
        return "accommodative"
    elif "neutral stance" in t:
        return "neutral"
    elif "tightening" in t or "withdrawal of accommodation" in t:
        return "hawkish"
    else:
        return "unknown"

full_doc_text = docs_df.groupby(["file_name", "start_date", "end_date"], as_index=False)["raw_text"].apply(lambda x: " ".join(x)).reset_index(drop=True)
full_doc_text["stance_label"] = full_doc_text["raw_text"].apply(infer_stance)

meeting_scores = scores_df.groupby(["file_name", "start_date", "end_date"], as_index=False).agg({
    "uncertainty_freq":"mean",
    "data_model_freq":"mean",
    "hawkish_freq":"mean",
    "dovish_freq":"mean",
    "uncertainty_embed":"mean",
    "data_model_embed":"mean",
    "hawkish_embed":"mean",
    "dovish_embed":"mean",
    "policy_bias_freq":"mean",
    "policy_bias_embed":"mean",
    "num_tokens":"sum"
})

meeting_scores = meeting_scores.merge(
    full_doc_text[["file_name", "start_date", "end_date", "stance_label"]],
    on=["file_name", "start_date", "end_date"],
    how="left"
)

# Create sortable timestamp using start_date
meeting_scores["start_date_ts"] = pd.to_datetime(meeting_scores["start_date"], format="%d/%m/%Y", errors="coerce")

# --------------------------------------------
# 12. STANDARDIZED INDICES
# --------------------------------------------
for col in ["uncertainty_freq", "data_model_freq", "uncertainty_embed", "data_model_embed", "policy_bias_freq", "policy_bias_embed"]:
    zcol = col + "_z"
    vals = meeting_scores[col].astype(float)
    meeting_scores[zcol] = (vals - vals.mean()) / (vals.std(ddof=0) if vals.std(ddof=0) != 0 else 1)

meeting_scores["composite_uncertainty_index"] = (
    0.5 * meeting_scores["uncertainty_freq_z"] +
    0.5 * meeting_scores["uncertainty_embed_z"]
)

meeting_scores["composite_data_model_uncertainty"] = (
    0.5 * meeting_scores["data_model_freq_z"] +
    0.5 * meeting_scores["data_model_embed_z"]
)

meeting_scores["composite_policy_bias"] = (
    0.5 * meeting_scores["policy_bias_freq_z"] +
    0.5 * meeting_scores["policy_bias_embed_z"]
)

# --------------------------------------------
# 13. SIMPLE IMPACT ANALYSIS
# --------------------------------------------
stance_map = {
    "hawkish": 1,
    "neutral": 0,
    "accommodative": -1,
    "unknown": np.nan
}
meeting_scores["stance_num"] = meeting_scores["stance_label"].map(stance_map)

reg_df = meeting_scores.dropna(subset=["stance_num", "composite_uncertainty_index", "composite_data_model_uncertainty"]).copy()

if len(reg_df) >= 5:
    X = reg_df[["composite_uncertainty_index", "composite_data_model_uncertainty"]]
    y = reg_df["stance_num"]

    reg = LinearRegression()
    reg.fit(X, y)

    coef_df = pd.DataFrame({
        "variable": X.columns,
        "coefficient": reg.coef_
    })
    coef_df["intercept"] = reg.intercept_
else:
    coef_df = pd.DataFrame(columns=["variable", "coefficient", "intercept"])

# --------------------------------------------
# 14. EXPORT TABLES AS XLSX
# --------------------------------------------
scores_df.to_excel(os.path.join(output_folder, "speaker_level_scores.xlsx"), index=False)
meeting_scores.to_excel(os.path.join(output_folder, "meeting_level_scores.xlsx"), index=False)
coef_df.to_excel(os.path.join(output_folder, "stance_regression_coefficients.xlsx"), index=False)

# --------------------------------------------
# 15. PLOTS
# --------------------------------------------
sns.set(style="whitegrid")

meeting_scores = meeting_scores.sort_values("start_date_ts")

plt.figure(figsize=(12, 5))
plt.plot(meeting_scores["start_date_ts"], meeting_scores["composite_uncertainty_index"], marker="o", label="Composite uncertainty")
plt.plot(meeting_scores["start_date_ts"], meeting_scores["composite_data_model_uncertainty"], marker="o", label="Data/model uncertainty")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.title("RBI MPC Minutes: Uncertainty Indices Over Time")
plt.xlabel("Meeting start date")
plt.ylabel("Standardized index")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "uncertainty_indices_over_time.png"), dpi=300)
plt.close()

plt.figure(figsize=(8, 6))
sns.regplot(
    data=meeting_scores,
    x="composite_uncertainty_index",
    y="composite_policy_bias",
    scatter_kws={"s": 70}
)
plt.title("Policy Bias vs Perceived Uncertainty")
plt.xlabel("Composite uncertainty index")
plt.ylabel("Composite policy bias")
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "policy_bias_vs_uncertainty.png"), dpi=300)
plt.close()

speaker_summary = scores_df.groupby("speaker", as_index=False).agg({
    "uncertainty_freq":"mean",
    "data_model_freq":"mean",
    "policy_bias_freq":"mean",
    "num_tokens":"sum"
})
speaker_summary = speaker_summary[speaker_summary["speaker"] != "FULL_DOCUMENT"].sort_values("uncertainty_freq", ascending=False)

if len(speaker_summary) > 0:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=speaker_summary, x="uncertainty_freq", y="speaker", palette="viridis")
    plt.title("Average Uncertainty Language by Speaker")
    plt.xlabel("Uncertainty-term frequency per 1000 tokens")
    plt.ylabel("Speaker")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "speaker_uncertainty_bar.png"), dpi=300)
    plt.close()

print("Done. Outputs saved to:", output_folder)

[nltk_data] Downloading package punkt to /Users/kalyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kalyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Found 59 PDF files.
Saved extracted sections.
Word2Vec model trained and saved.
Done. Outputs saved to: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/output/mpc


## Simplifying code to extract Minutes

In [1]:
# ============================================
# RBI MPC Minutes: comprehensive parser
# with improved member parsing, fixed institutional ordering,
# repo extraction, and resolution extraction
# ============================================

# PLAIN ENGLISH EXPLANATION
# ------------------------------------------------------------
# This code reads RBI MPC minutes PDF files from a folder and
# extracts a structured dataset for each meeting.
#
# It does the following:
# 1. Reads each PDF and extracts raw text.
# 2. Extracts meeting start_date and end_date from the minutes header.
# 3. Handles date formats such as:
#       - April 6 to 8, 2022
#       - August 5-7, 2019
#       - May 2 and 4, 2022
#       - March 24, 26 and 27, 2020
#       - September 29, 30 and October 1, 2025
#       - September 29 to October 1, 2025
# 4. Extracts MPC member names and designations.
# 5. Uses semicolons to separate members, while commas remain
#    part of each member's designation.
# 6. Replaces statutory nominee text such as:
#    "the officer of the Bank nominated by the Central Board under
#    Section 45ZB(2)(c) of the amended Act"
#    with "RBI".
# 7. Treats the text after "and was chaired by" as the sixth member.
#    The name is taken from before the comma, and the designation
#    is taken from after the comma.
# 8. Extracts all "Statement by ..." sections.
# 9. Uses fuzzy name matching so that small differences in names
#    across sections do not break the dataset.
# 10. Extracts the repo rate decision and the broad policy action
#     (hike, cut, unchanged), including the exact RBI wording
#     "keep the policy repo rate under the liquidity adjustment
#     facility (LAF) unchanged at 6.50 per cent".
# 11. Extracts the policy stance in the resolution and the broad
#     stance action (continue / withdraw / change).
# 12. Extracts explicit rate votes where available.
# 13. Infers stance votes from statements where possible.
# 14. Outputs the six members in a fixed institutional order:
#       mem_1 to mem_3 = three external members
#       mem_4          = RBI nominee
#       mem_5          = Deputy Governor in charge of monetary policy
#       mem_6          = Governor / Chair
# 15. Writes one Excel workbook with multiple sheets.
# 16. Writes a parser_debug sheet to flag likely extraction issues.

import os
import re
import glob
import calendar
import warnings
import time
from datetime import datetime
from difflib import SequenceMatcher

import fitz
import pandas as pd
from dateutil import parser as dtparser

# --------------------------------------------
# PRINT START TIME
# --------------------------------------------
code_start_time = datetime.now()
timer_start = time.time()
print("Code run start time:", code_start_time.strftime("%d/%m/%Y %H:%M:%S"))

warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"
pdf_folder = os.path.join(proj_base, "data/mpc-minutes")
output_folder = os.path.join(proj_base, "output/mpc")
os.makedirs(output_folder, exist_ok=True)

# --------------------------------------------
# TEXT HELPERS
# --------------------------------------------
def normalize_text(text):
    if text is None:
        return ""
    text = text.replace("\x0c", " ")
    text = text.replace("\u00a0", " ")
    text = text.replace("\r", "\n")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def flatten_text(text):
    t = normalize_text(text)
    t = re.sub(r"\n+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def extract_text_from_pdf(pdf_path):
    text_parts = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text_parts.append(page.get_text("text"))
    doc.close()
    return "\n".join(text_parts)

def normalize_rbi_statutory_phrase(text):
    if text is None:
        return None

    patterns = [
        r"\(?\s*the officer of the Reserve Bank nominated by the Central Board under Section 45ZB\(2\)\(c\) of the Reserve Bank of India Act,\s*1934\s*\)?",
        r"\(?\s*the officer of the Bank nominated by the Central Board under Section 45ZB\(2\)\(c\) of the amended Act\s*\)?",
        r"\(?\s*the officer of the Bank nominated by the Central Board under Section 45ZB\(2\)\(c\)\s*of\s*the amended Act\s*\)?",
        r"\(?\s*the officer of the Bank nominated by the Central Board under Section 45ZB\(2\)\(c\).*?\)?"
    ]

    out = str(text)
    for pat in patterns:
        out = re.sub(pat, ",RBI", out, flags=re.IGNORECASE)

    out = re.sub(r"\(\s*RBI\s*\)", ",RBI", out, flags=re.IGNORECASE)
    out = re.sub(r"\s+", " ", out).strip(" ,;")
    return out

def clean_name(name):
    if name is None:
        return None
    name = re.sub(r"\s+", " ", str(name)).strip(" \n\t-;,.")
    return name if name else None

def clean_designation(desig):
    if desig is None:
        return None
    desig = normalize_rbi_statutory_phrase(str(desig))
    desig = re.sub(r"\s+", " ", desig).strip(" \n\t-;,.")
    return desig if desig else None

# --------------------------------------------
# DATE HELPERS
# --------------------------------------------
MONTHS = {m.lower(): i for i, m in enumerate(calendar.month_name) if m}
MONTHS_SHORT = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def month_to_num(month_str):
    if not month_str:
        return None
    x = month_str.strip().lower().rstrip(".")
    if x in MONTHS:
        return MONTHS[x]
    if x in MONTHS_SHORT:
        return MONTHS_SHORT[x]
    return None

def format_dt(day, month_num, year):
    return pd.Timestamp(year=int(year), month=int(month_num), day=int(day)).strftime("%d/%m/%Y")

def extract_explicit_year(text, min_year=2015, max_year=2035):
    years = re.findall(r"\b(19\d{2}|20\d{2})\b", text)
    years = [int(y) for y in years if min_year <= int(y) <= max_year]
    if not years:
        return None
    return years[-1]

# --------------------------------------------
# DATE EXTRACTION
# --------------------------------------------
def extract_mpc_date_range(text):
    t = flatten_text(text)
    t_short = t[:2500]

    header_candidates = []

    pats = [
        r"(Minutes of the Monetary Policy Committee Meeting,\s*.+?)(?:\[|$)",
        r"(Minutes of the Monetary Policy Committee Meeting\s*.+?)(?:\[|$)",
        r"(The .*? Monetary Policy Committee .*? was held during .*?)(?:Resolution|$)",
        r"(The .*? Monetary Policy Committee .*? was held from .*?)(?:Resolution|$)",
        r"(held during .*?)(?:Resolution|$)",
        r"(held from .*?)(?:Resolution|$)"
    ]

    for pat in pats:
        m = re.search(pat, t_short, flags=re.IGNORECASE)
        if m:
            header_candidates.append(m.group(1).strip())

    if not header_candidates:
        header_candidates = [t_short]

    for candidate in header_candidates:
        c = flatten_text(candidate)
        explicit_year = extract_explicit_year(c)

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2})\s*-\s*(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, d2, year = m.groups()
            try:
                m1 = month_to_num(month1)
                return format_dt(d1, m1, year), format_dt(d2, m1, year), "same_month_hyphen_range", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2})\s+(?:to|-)\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, d2, year = m.groups()
            try:
                m1 = month_to_num(month1)
                return format_dt(d1, m1, year), format_dt(d2, m1, year), "same_month_range", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2})\s+and\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, d2, year = m.groups()
            try:
                m1 = month_to_num(month1)
                return format_dt(d1, m1, year), format_dt(d2, m1, year), "same_month_two_days", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2}),\s*(\d{1,2})\s+and\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, d2, d3, year = m.groups()
            try:
                m1 = month_to_num(month1)
                return format_dt(d1, m1, year), format_dt(d3, m1, year), "same_month_three_days", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2}),\s*(\d{1,2})\s+and\s+([A-Za-z]+)\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, d2, month2, d3, year = m.groups()
            try:
                m1 = month_to_num(month1)
                m2 = month_to_num(month2)
                return format_dt(d1, m1, year), format_dt(d3, m2, year), "cross_month_three_days", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2})\s*-\s*([A-Za-z]+)\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, month2, d2, year = m.groups()
            try:
                m1 = month_to_num(month1)
                m2 = month_to_num(month2)
                return format_dt(d1, m1, year), format_dt(d2, m2, year), "cross_month_hyphen_range", c
            except:
                pass

        m = re.search(r"([A-Za-z]+)\s+(\d{1,2})\s+(?:to|-)\s+([A-Za-z]+)\s+(\d{1,2}),\s*(19\d{2}|20\d{2})", c, flags=re.IGNORECASE)
        if m:
            month1, d1, month2, d2, year = m.groups()
            try:
                m1 = month_to_num(month1)
                m2 = month_to_num(month2)
                return format_dt(d1, m1, year), format_dt(d2, m2, year), "cross_month_range", c
            except:
                pass

        if explicit_year is not None:
            try:
                dt = dtparser.parse(c, fuzzy=True, default=datetime(explicit_year, 1, 1))
                month_num = dt.month
                day_num = dt.day
                d = pd.Timestamp(year=explicit_year, month=month_num, day=day_num).strftime("%d/%m/%Y")
                return d, d, "dateutil_forced_year_fallback", c
            except:
                pass

    return None, None, "no_match", header_candidates[0] if header_candidates else None

# --------------------------------------------
# NAME CANONICALIZATION + FUZZY MATCHING
# --------------------------------------------
TITLE_PATTERN = r"(?:Dr\.|Prof\.|Professor|Shri|Smt\.?|Mr\.?|Mrs\.?|Ms\.?)"
NAME_PATTERN = r"[A-Z][A-Za-z\.]*(?:\s+[A-Z][A-Za-z\.]*)+"

def canonicalize_name(name):
    if not name:
        return ""
    x = name.lower().strip()
    x = re.sub(r"\bprofessor\b", "prof", x)
    x = re.sub(r"\bprof\.\b", "prof", x)
    x = re.sub(r"\bdr\.\b", "dr", x)
    x = re.sub(r"\bshri\b", "", x)
    x = re.sub(r"\bsmt\.?\b", "", x)
    x = re.sub(r"\bmr\.?\b", "", x)
    x = re.sub(r"\bmrs\.?\b", "", x)
    x = re.sub(r"\bms\.?\b", "", x)
    x = re.sub(r"[^a-z\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def similarity(a, b):
    return SequenceMatcher(None, canonicalize_name(a), canonicalize_name(b)).ratio()

def best_fuzzy_match(name, candidates, threshold=0.82):
    if not name or not candidates:
        return None

    exact_canon = canonicalize_name(name)
    canon_map = {c: canonicalize_name(c) for c in candidates}

    for c, cc in canon_map.items():
        if exact_canon == cc:
            return c

    scored = [(c, similarity(name, c)) for c in candidates]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    if scored and scored[0][1] >= threshold:
        return scored[0][0]

    return None

def reconcile_name_dict(source_dict, target_names, threshold=0.82):
    out = {}
    for src_name, value in source_dict.items():
        match = best_fuzzy_match(src_name, target_names, threshold=threshold)
        final_key = match if match else src_name
        if final_key not in out or (out[final_key] is None and value is not None):
            out[final_key] = value
    return out

# --------------------------------------------
# MEMBER / DESIGNATION EXTRACTION
# --------------------------------------------
def extract_attendance_block(text):
    t = normalize_text(text)

    patterns = [
        r"The meeting was attended by all the members\s*-\s*(.+?)\s*-\s*and was chaired by\s*(.+?)\.",
        r"The meeting was attended by all the members\s*-\s*(.+?)\s*and was chaired by\s*(.+?)\.",
        r"The meeting was attended by all the members\s*[–-]\s*(.+?)\s*[–-]\s*and was chaired by\s*(.+?)\.",
        r"The meeting was attended by all the members\s*[–-]\s*(.+?)\s*and was chaired by\s*(.+?)\.",
        r"The meeting was chaired by\s*(.+?)\s*and was attended by all the members\s*-\s*(.+?)\.",
        r"The MPC members\s+(.+?)\s+attended the meeting\."
    ]

    for idx, pat in enumerate(patterns):
        m = re.search(pat, t, flags=re.IGNORECASE | re.DOTALL)
        if m:
            if idx in [0, 1, 2, 3]:
                return m.group(1).strip(), m.group(2).strip()
            elif idx == 4:
                return m.group(2).strip(), m.group(1).strip()
            else:
                return m.group(1).strip(), None

    return None, None

def split_member_entries(member_block):
    if not member_block:
        return []

    block = normalize_rbi_statutory_phrase(member_block)
    block = re.sub(r"\s+", " ", block).strip()

    parts = [p.strip(" ;") for p in re.split(r"\s*;\s*", block) if p.strip()]
    return parts

def parse_name_designation(entry):
    entry = normalize_rbi_statutory_phrase(entry)
    entry = re.sub(r"\s+", " ", entry).strip(" ;,.-")

    pat = rf"^({TITLE_PATTERN}\s+{NAME_PATTERN})(?:,\s*(.+))?$"
    m = re.match(pat, entry, flags=re.IGNORECASE)

    if m:
        return clean_name(m.group(1)), clean_designation(m.group(2))

    if "," in entry:
        left, right = entry.split(",", 1)
        return clean_name(left), clean_designation(right)

    return clean_name(entry), None

def extract_chair_name_designation(chair_block):
    if not chair_block:
        return None, None

    chair_block = normalize_rbi_statutory_phrase(chair_block)
    chair_block = re.sub(r"\s+", " ", chair_block).strip(" ;,.-")

    # Required behavior:
    # before the comma = name
    # after the comma  = designation
    if "," in chair_block:
        left, right = chair_block.split(",", 1)
        return clean_name(left), clean_designation(right)

    m = re.match(rf"^({TITLE_PATTERN}\s+{NAME_PATTERN})(?:,\s*(.+))?$", chair_block, flags=re.IGNORECASE)
    if m:
        return clean_name(m.group(1)), clean_designation(m.group(2))

    return clean_name(chair_block), None

def extract_statement_members(text):
    t = normalize_text(text)
    pattern = rf"Statement by\s+({TITLE_PATTERN}\s+{NAME_PATTERN})\s*(?:\n|$)"
    matches = re.findall(pattern, t, flags=re.IGNORECASE)
    names = []
    for x in matches:
        nm = clean_name(x)
        if nm and nm not in names:
            names.append(nm)
    return names

def extract_member_names_from_mpc_members_line(text):
    t = normalize_text(text)

    patterns = [
        r"The MPC members\s+(.+?)\s+attended the meeting\.",
        r"MPC members\s+(.+?)\s+attended the meeting\.",
        r"The Monetary Policy Committee .*? members\s+(.+?)\s+attended the meeting\."
    ]

    found = []

    for pat in patterns:
        for m in re.finditer(pat, t, flags=re.IGNORECASE | re.DOTALL):
            block = normalize_rbi_statutory_phrase(m.group(1))
            block = re.sub(r"\s+", " ", block).strip()
            block = re.sub(r"\s+and\s+", "; ", block)
            parts = [p.strip(" ;,.") for p in re.split(r"\s*;\s*", block) if p.strip()]
            for part in parts:
                found.append(part)

    cleaned = []
    seen = set()
    for x in found:
        name, _ = parse_name_designation(x)
        if name and name not in seen:
            cleaned.append(name)
            seen.add(name)

    return cleaned

def extract_all_members_with_designations(text):
    member_block, chair_block = extract_attendance_block(text)

    members = []
    seen = set()

    entries = split_member_entries(member_block)
    for entry in entries:
        name, designation = parse_name_designation(entry)
        if name and name not in seen:
            members.append({"name": name, "designation": designation})
            seen.add(name)

    # Explicitly treat chair block as sixth member
    chair_name, chair_designation = extract_chair_name_designation(chair_block)
    if chair_name:
        if chair_name not in seen:
            members.append({"name": chair_name, "designation": chair_designation})
            seen.add(chair_name)
        else:
            for m in members:
                if m["name"] == chair_name and not m.get("designation") and chair_designation:
                    m["designation"] = chair_designation

    mpc_member_names = extract_member_names_from_mpc_members_line(text)
    for nm in mpc_member_names:
        if nm not in seen:
            members.append({"name": nm, "designation": None})
            seen.add(nm)

    stmt_names = extract_statement_members(text)
    for nm in stmt_names:
        if nm not in seen:
            members.append({"name": nm, "designation": None})
            seen.add(nm)

    return members[:6]

# --------------------------------------------
# MEMBER ORDERING
# --------------------------------------------
def classify_member_role(member):
    if member is None:
        return "unknown"

    designation = (member.get("designation") or "").lower()
    name = (member.get("name") or "").lower()

    if "governor" in designation or "chair" in designation:
        return "governor"

    if "deputy governor" in designation and "monetary policy" in designation:
        return "deputy_governor_mp"

    if designation == "rbi":
        return "rbi_nominee"

    if "executive director" in designation and "rbi" in designation:
        return "rbi_nominee"

    if "executive director" in designation and "reserve bank" in designation:
        return "rbi_nominee"

    if "reserve bank" in designation and "nominated" in designation:
        return "rbi_nominee"

    if designation.startswith("rbi"):
        return "rbi_nominee"

    return "external"

def order_members_institutionally(member_info):
    externals = []
    rbi_nominee = None
    deputy_governor = None
    governor = None
    leftovers = []

    for m in member_info:
        role = classify_member_role(m)

        if role == "governor":
            if governor is None:
                governor = m
            else:
                leftovers.append(m)

        elif role == "deputy_governor_mp":
            if deputy_governor is None:
                deputy_governor = m
            else:
                leftovers.append(m)

        elif role == "rbi_nominee":
            if rbi_nominee is None:
                rbi_nominee = m
            else:
                leftovers.append(m)

        elif role == "external":
            if len(externals) < 3:
                externals.append(m)
            else:
                leftovers.append(m)
        else:
            leftovers.append(m)

    for m in leftovers:
        if len(externals) < 3:
            externals.append(m)
        elif rbi_nominee is None:
            rbi_nominee = m
        elif deputy_governor is None:
            deputy_governor = m
        elif governor is None:
            governor = m

    ordered = []
    ordered.extend(externals[:3])
    while len(ordered) < 3:
        ordered.append({"name": None, "designation": None})

    ordered.append(rbi_nominee if rbi_nominee is not None else {"name": None, "designation": None})
    ordered.append(deputy_governor if deputy_governor is not None else {"name": None, "designation": None})
    ordered.append(governor if governor is not None else {"name": None, "designation": None})

    return ordered[:6]

# --------------------------------------------
# STATEMENT EXTRACTION
# --------------------------------------------
def extract_member_statements(text):
    t = normalize_text(text)
    pattern = rf"Statement by\s+({TITLE_PATTERN}\s+{NAME_PATTERN})\s*(?:\n|$)"
    matches = list(re.finditer(pattern, t, flags=re.IGNORECASE))
    out = {}

    for i, m in enumerate(matches):
        name = clean_name(m.group(1))
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(t)
        stmt = t[start:end].strip()
        stmt = re.sub(r"\n\s*\d+\s*$", "", stmt).strip()
        out[name] = stmt

    return out

# --------------------------------------------
# POLICY RATE / ACTION / STANCE
# --------------------------------------------
def extract_policy_rate_and_action(text):
    t = flatten_text(text).lower()

    patterns = [
        # exact RBI unchanged wording
        (
            r"keep\s+the\s+policy\s+repo\s+rate\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?\s+unchanged\s+at\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "unchanged"
        ),
        (
            r"(?:kept|maintain|maintained|retain|retained)\s+the\s+policy\s+repo\s+rate\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?\s+unchanged\s+at\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "unchanged"
        ),
        # broader unchanged forms
        (
            r"(?:keep|kept|maintain|maintained|retain|retained)\s+the\s+policy\s+repo\s+rate(?:\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?)?.*?unchanged\s+at\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "unchanged"
        ),
        # hikes
        (
            r"(?:increase|increased|raise|raised)\s+the\s+policy\s+repo\s+rate(?:\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?)?.*?to\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "hike"
        ),
        # cuts
        (
            r"(?:reduce|reduced|cut|lower)\s+the\s+policy\s+repo\s+rate(?:\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?)?.*?to\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "cut"
        ),
        # kept repo unchanged generic
        (
            r"repo\s+rate(?:\s+under\s+the\s+liquidity\s+adjustment\s+facility\s*\(?laf\)?)?\s+unchanged\s+at\s+(\d+(?:\.\d+)?)\s+per\s+cent",
            "unchanged"
        ),
    ]

    for pat, action in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            rate = float(m.group(1))
            return rate, action, m.group(0)

    return None, None, None

def extract_policy_stance_action_resolution(text):
    t = flatten_text(text).lower()

    patterns = [
        (
            r"(change|changed|shift|shifted|revised|revised the)\s+(?:the\s+)?(?:monetary\s+policy\s+)?stance\s+from\s+"
            r"(accommodative|neutral|calibrated\s+tightening|withdrawal\s+of\s+accommodation)\s+to\s+"
            r"(accommodative|neutral|calibrated\s+tightening|withdrawal\s+of\s+accommodation)",
            "change_pattern"
        ),
        (
            r"(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining|keep|kept|remain|remained|remaining|focus|focused)\s+"
            r"(?:with\s+)?(?:the\s+)?(?:stance\s+of\s+)?(?:being\s+)?(?:remain\s+)?(?:focused\s+on\s+)?"
            r"(withdrawal\s+of\s+accommodation)",
            "withdrawal_of_accommodation"
        ),
        (
            r"\b(withdrawal\s+of\s+accommodation)\b",
            "withdrawal_of_accommodation_bare"
        ),
        (
            r"(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining|keep|kept|adopt|adopted)\s+"
            r"(?:with\s+)?(?:the\s+)?(calibrated\s+tightening)\s+stance",
            "calibrated_tightening"
        ),
        (
            r"\b(calibrated\s+tightening)\s+stance\b",
            "calibrated_tightening_bare"
        ),
        (
            r"(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining|keep|kept)\s+"
            r"(?:with\s+)?(?:an?\s+)?(?:the\s+)?(accommodative)\s+stance(?:\s+of\s+monetary\s+policy)?",
            "accommodative"
        ),
        (
            r"(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining|keep|kept)\s+"
            r"(?:with\s+)?(?:an?\s+)?(?:the\s+)?(neutral)\s+stance(?:\s+of\s+monetary\s+policy)?",
            "neutral"
        ),
        (
            r"(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining)\s+(accommodation)",
            "accommodation_word"
        ),
        (
            r"(consistent\s+with\s+an?\s+(accommodative|neutral|calibrated\s+tightening)\s+stance\s+of\s+monetary\s+policy)",
            "consistent_pattern"
        ),
        (
            r"(adopt|adopted)\s+(?:an?\s+)?(accommodative|neutral)\s+stance(?:\s+of\s+monetary\s+policy)?",
            "adopt_pattern"
        ),
        (
            r"\b(accommodative|neutral)\s+stance(?:\s+of\s+monetary\s+policy)?\b",
            "bare_simple_stance"
        ),
    ]

    for pat, label in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if not m:
            continue

        matched_text = m.group(0).strip()

        if label == "change_pattern":
            new_stance = m.group(3).strip().lower()
            return new_stance, "change", matched_text

        if label in ["withdrawal_of_accommodation", "withdrawal_of_accommodation_bare"]:
            if re.search(r"\b(continue|continued|continuing|maintain|maintained|maintaining|retain|retained|retaining|keep|kept|remain|remained|remaining|focus|focused)\b", matched_text):
                return "withdrawal of accommodation", "continue", matched_text
            return "withdrawal of accommodation", "withdraw", matched_text

        if label in ["calibrated_tightening", "calibrated_tightening_bare"]:
            if re.search(r"\b(adopt|adopted)\b", matched_text):
                return "calibrated tightening", "change", matched_text
            return "calibrated tightening", "continue", matched_text

        if label == "accommodative":
            return "accommodative", "continue", matched_text

        if label == "neutral":
            return "neutral", "continue", matched_text

        if label == "accommodation_word":
            return "accommodative", "continue", matched_text

        if label == "consistent_pattern":
            stance = m.group(2).strip().lower()
            return stance, "continue", matched_text

        if label == "adopt_pattern":
            stance = m.group(2).strip().lower()
            return stance, "change", matched_text

        if label == "bare_simple_stance":
            stance = m.group(1).strip().lower()
            return stance, "continue", matched_text

    return None, None, None

# --------------------------------------------
# RATE VOTE EXTRACTION
# --------------------------------------------
def extract_rate_vote_map(text):
    t = normalize_text(text)
    vote_map = {}

    patterns = [
        r"Voting on the Resolution to .*?policy repo rate.*?\n(.*?)(?:Statement by|$)",
        r"Member\s+Vote\s+(.*?)(?:Statement by|$)"
    ]

    block = None
    for pat in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE | re.DOTALL)
        if m:
            block = m.group(1)
            break

    if not block:
        return vote_map

    lines = [re.sub(r"\s+", " ", x).strip() for x in block.split("\n")]
    lines = [x for x in lines if x]

    i = 0
    while i < len(lines) - 1:
        a = lines[i]
        b = lines[i + 1]
        if re.match(rf"^({TITLE_PATTERN})\s+", a) and b.lower() in ["yes", "no"]:
            vote_map[clean_name(a)] = b.title()
            i += 2
        else:
            i += 1

    return vote_map

# --------------------------------------------
# INFER VOTES FROM STATEMENTS
# --------------------------------------------
def infer_rate_vote_from_statement(stmt):
    if stmt is None or str(stmt).strip() == "":
        return None

    t = flatten_text(stmt).lower()

    if re.search(r"i vote.*?(keep|maintain|continue).*?repo rate.*?unchanged", t):
        return "unchanged"
    if re.search(r"i vote.*?(increase|raise|hike).*?repo rate", t):
        return "hike"
    if re.search(r"i vote.*?(reduce|cut|lower).*?repo rate", t):
        return "cut"

    return None

def infer_stance_vote_from_statement(stmt):
    if stmt is None or str(stmt).strip() == "":
        return None

    t = flatten_text(stmt).lower()

    patterns = [
        (r"i also vote to maintain the accommodative stance as long as necessary", "maintain accommodative stance"),
        (r"i vote to continue with the accommodative stance", "continue accommodative stance"),
        (r"i voted to continue with the accommodative stance", "continue accommodative stance"),
        (r"i vote to remain accommodative while focusing on withdrawal of accommodation", "remain accommodative while focusing on withdrawal of accommodation"),
        (r"i vote to remain focused on withdrawal of accommodation", "remain focused on withdrawal of accommodation"),
        (r"i vote for a neutral stance", "neutral stance"),
        (r"i vote to maintain a neutral stance", "neutral stance"),
        (r"i voted for a neutral stance", "neutral stance"),
        (r"i vote for an accommodative stance", "accommodative stance"),
        (r"i vote to change the stance from neutral to accommodative", "change stance from neutral to accommodative"),
        (r"i support the stance of (.+?)(?:\.|;)", None),
        (r"i vote in favour of (.+?stance.+?)(?:\.|;)", None),
        (r"i vote for (.+?stance.+?)(?:\.|;)", None),
    ]

    for pat, label in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            if label is not None:
                return label
            return clean_designation(m.group(1))

    return None

# --------------------------------------------
# SINGLE MEETING PARSER
# --------------------------------------------
def parse_single_meeting(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)

    start_date, end_date, matched_rule, candidate_text = extract_mpc_date_range(raw_text)

    member_info = extract_all_members_with_designations(raw_text)
    member_info = order_members_institutionally(member_info)

    master_names = [x["name"] for x in member_info if x["name"]]

    stmt_map_raw = extract_member_statements(raw_text)
    rate_vote_map_raw = extract_rate_vote_map(raw_text)

    stmt_map = reconcile_name_dict(stmt_map_raw, master_names, threshold=0.82)
    rate_vote_map = reconcile_name_dict(rate_vote_map_raw, master_names, threshold=0.82)

    policy_rate, policy_action, rate_sentence = extract_policy_rate_and_action(raw_text)
    policy_stance, policy_stance_action, stance_sentence = extract_policy_stance_action_resolution(raw_text)

    row = {
        "file_name": os.path.basename(pdf_path),
        "start_date": start_date,
        "end_date": end_date,
        "res_pol_rate_repo": policy_rate,
        "res_pol_rate_action": policy_action,
        "res_pol_stance": policy_stance,
        "res_pol_stance_action": policy_stance_action,
        "matched_rate_sentence": rate_sentence,
        "matched_stance_sentence": stance_sentence
    }

    member_rows = []

    for i in range(6):
        if i < len(member_info):
            mem = member_info[i]["name"]
            designation = member_info[i]["designation"]
        else:
            mem = None
            designation = None

        stmt = stmt_map.get(mem) if mem else None

        rate_vote = rate_vote_map.get(mem)
        if rate_vote is None:
            rate_vote = infer_rate_vote_from_statement(stmt)

        stance_vote = infer_stance_vote_from_statement(stmt)

        row[f"mem_{i+1}"] = mem
        row[f"designation_mem_{i+1}"] = designation
        row[f"rate_vote_mem_{i+1}"] = rate_vote
        row[f"stance_vote_mem_{i+1}"] = stance_vote
        row[f"stmt_mem_{i+1}"] = stmt

        member_rows.append({
            "file_name": os.path.basename(pdf_path),
            "start_date": start_date,
            "end_date": end_date,
            "member_slot": f"mem_{i+1}",
            "member_name": mem,
            "designation": designation,
            "rate_vote": rate_vote,
            "stance_vote": stance_vote,
            "statement": stmt,
            "res_pol_rate_repo": policy_rate,
            "res_pol_rate_action": policy_action,
            "res_pol_stance": policy_stance,
            "res_pol_stance_action": policy_stance_action
        })

    date_debug_row = {
        "file_name": os.path.basename(pdf_path),
        "start_date": start_date,
        "end_date": end_date,
        "matched_rule": matched_rule,
        "candidate_text": candidate_text
    }

    return row, member_rows, date_debug_row

# --------------------------------------------
# RUN ALL FILES
# --------------------------------------------
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

meeting_rows = []
member_rows_all = []
date_debug_rows = []

for pdf_path in pdf_files:
    row, member_rows, date_debug_row = parse_single_meeting(pdf_path)
    meeting_rows.append(row)
    member_rows_all.extend(member_rows)
    date_debug_rows.append(date_debug_row)

meetings_df = pd.DataFrame(meeting_rows)
members_df = pd.DataFrame(member_rows_all)
date_debug_df = pd.DataFrame(date_debug_rows)

meetings_df["_sort_date"] = pd.to_datetime(meetings_df["start_date"], format="%d/%m/%Y", errors="coerce")
meetings_df = meetings_df.sort_values(["_sort_date", "file_name"]).drop(columns=["_sort_date"])

members_df["_sort_date"] = pd.to_datetime(members_df["start_date"], format="%d/%m/%Y", errors="coerce")
members_df = members_df.sort_values(["_sort_date", "file_name", "member_slot"]).drop(columns=["_sort_date"])

date_debug_df["_sort_date"] = pd.to_datetime(date_debug_df["start_date"], format="%d/%m/%Y", errors="coerce")
date_debug_df = date_debug_df.sort_values(["_sort_date", "file_name"]).drop(columns=["_sort_date"])

# --------------------------------------------
# PARSER DEBUG SHEET
# --------------------------------------------
def build_parser_debug(meetings_df, date_debug_df):
    dbg = meetings_df.merge(
        date_debug_df[["file_name", "matched_rule", "candidate_text"]],
        on="file_name",
        how="left"
    ).copy()

    mem_cols = [f"mem_{i}" for i in range(1, 7)]
    dbg["member_count_found"] = dbg[mem_cols].notna().sum(axis=1)
    dbg["missing_member_slots"] = dbg[mem_cols].isna().sum(axis=1)

    dbg["flag_missing_repo_rate"] = dbg["res_pol_rate_repo"].isna()
    dbg["flag_missing_repo_action"] = dbg["res_pol_rate_action"].isna()
    dbg["flag_missing_stance"] = dbg["res_pol_stance"].isna()
    dbg["flag_missing_stance_action"] = dbg["res_pol_stance_action"].isna()
    dbg["flag_less_than_6_members"] = dbg["member_count_found"] < 6
    dbg["flag_date_no_match"] = dbg["matched_rule"].fillna("no_match").eq("no_match")
    dbg["flag_mem4_not_rbi_nominee"] = ~dbg["designation_mem_4"].fillna("").str.lower().str.contains("rbi|reserve bank|executive director", regex=True)
    dbg["flag_mem5_not_dg_mp"] = ~dbg["designation_mem_5"].fillna("").str.lower().str.contains("deputy governor", regex=True)
    dbg["flag_mem6_not_governor"] = ~dbg["designation_mem_6"].fillna("").str.lower().str.contains("governor|chair", regex=True)

    dbg["needs_manual_review"] = (
        dbg["flag_missing_repo_rate"] |
        dbg["flag_missing_repo_action"] |
        dbg["flag_missing_stance"] |
        dbg["flag_missing_stance_action"] |
        dbg["flag_less_than_6_members"] |
        dbg["flag_date_no_match"] |
        dbg["flag_mem4_not_rbi_nominee"] |
        dbg["flag_mem5_not_dg_mp"] |
        dbg["flag_mem6_not_governor"]
    )

    debug_cols = [
        "file_name",
        "start_date",
        "end_date",
        "matched_rule",
        "member_count_found",
        "missing_member_slots",
        "res_pol_rate_repo",
        "res_pol_rate_action",
        "res_pol_stance",
        "res_pol_stance_action",
        "matched_rate_sentence",
        "matched_stance_sentence",
        "flag_missing_repo_rate",
        "flag_missing_repo_action",
        "flag_missing_stance",
        "flag_missing_stance_action",
        "flag_less_than_6_members",
        "flag_date_no_match",
        "flag_mem4_not_rbi_nominee",
        "flag_mem5_not_dg_mp",
        "flag_mem6_not_governor",
        "needs_manual_review",
        "candidate_text"
    ] + mem_cols

    dbg = dbg[debug_cols].sort_values(
        by=["needs_manual_review", "start_date", "file_name"],
        ascending=[False, True, True]
    )

    return dbg

parser_debug_df = build_parser_debug(meetings_df, date_debug_df)

# --------------------------------------------
# METADATA
# --------------------------------------------
meta_df = pd.DataFrame({
    "field": [
        "source_folder",
        "output_folder",
        "total_pdf_files",
        "date_parser",
        "name_matcher",
        "name_match_threshold",
        "member_ordering",
        "run_start_time"
    ],
    "value": [
        pdf_folder,
        output_folder,
        len(pdf_files),
        "custom regex + fuzzy fallback via dateutil with explicit-year guard",
        "difflib.SequenceMatcher",
        0.82,
        "mem_1-3 external; mem_4 RBI nominee; mem_5 Deputy Governor MP; mem_6 Governor/Chair",
        code_start_time.strftime("%d/%m/%Y %H:%M:%S")
    ]
})

# --------------------------------------------
# SAVE TO XLSX
# --------------------------------------------
main_xlsx = os.path.join(output_folder, "rbi_mpc_votes_and_statements.xlsx")

with pd.ExcelWriter(main_xlsx, engine="openpyxl") as writer:
    meetings_df.to_excel(writer, sheet_name="meeting_level", index=False)
    members_df.to_excel(writer, sheet_name="member_level_long", index=False)
    meta_df.to_excel(writer, sheet_name="metadata", index=False)
    date_debug_df.to_excel(writer, sheet_name="date_debug", index=False)
    parser_debug_df.to_excel(writer, sheet_name="parser_debug", index=False)

print("Saved:", main_xlsx)

# --------------------------------------------
# PRINT END TIME
# --------------------------------------------
code_end_time = datetime.now()
timer_end = time.time()

print("Code run end time:", code_end_time.strftime("%d/%m/%Y %H:%M:%S"))
print("Elapsed time (seconds):", round(timer_end - timer_start, 2))

Code run start time: 18/05/2026 11:00:22
Saved: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/output/mpc/rbi_mpc_votes_and_statements.xlsx
Code run end time: 18/05/2026 11:05:46
Elapsed time (seconds): 324.3
